<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_02_4_pytorch_class_sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks

**Module 2: PyTorch for Neural Networks**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 2 Material

* Part 2.1: Numeric Processing with PyTorch [[Video]]() [[Notebook]](t81_558_class_02_1_pytorch_numerical.ipynb)
* Part 2.2: Deep Learning with PyTorch [[Video]]() [[Notebook]](t81_558_class_02_2_pytorch_neural.ipynb)
* Part 2.3: Preprocessing for PyTorch [[Video]]() [[Notebook]](t81_558_class_02_3_feature_encode.ipynb)
* **Part 2.4: nn.Module vs nn.Sequential for PyTorch** [[Video]]() [[Notebook]](t81_558_class_02_4_pytorch_class_sequence.ipynb)
* Part 2.5: Beyond the CPU[[Video]]() [[Notebook]](t81_558_class_02_5_beyond_cpu.ipynb)

# Google Colab Instructions

The following code ensures that Google Colab is running and maps Google Drive if needed.

In [1]:
try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False
    import torch

# Make use of a GPU or MPS (Apple) if one is available.  (see module 3.2)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: using Google CoLab
Using device: cuda


# Part 2.4: Sequences vs Classes in PyTorch

Previously, we explored the fundamentals of PyTorch and how to define neural networks using the `nn.Sequential` module. This approach allowed us to build neural networks by specifying a sequence of layers in a straightforward, concise manner. While `nn.Sequential` is a method we will use extensively in this course due to its simplicity, it is essential to understand an alternative way of defining neural networks as a class.

## Defining Neural Networks as Sequences

Before we review the class-based approach, let's recap how we define neural networks using the `nn.Sequential` module. In PyTorch, many neural networks can be represented as a sequence of layers. Each layer performs a specific computation on the input data and passes the transformed output to the next layer.

Here's a quick example of defining a simple neural network using `nn.Sequential`:

```python
import torch
import torch.nn as nn

# Define the neural network architecture
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)
```

In this example, we created a neural network with three linear layers interspersed with ReLU activation functions. The final layer produces raw outputs (logits), which are typically passed to a loss function such as `nn.CrossEntropyLoss` during training.

## Defining Neural Networks as Classes

While the `nn.Sequential` approach is intuitive and convenient for many cases, it can become limiting when we need more flexibility in the network architecture. Defining neural networks as classes allows us to create custom architectures with complex behaviors and reusable components.

To define a neural network as a class, we typically subclass the `nn.Module` class provided by PyTorch. This base class organizes the network's parameters and defines the forward computation, while gradient calculations are handled automatically by PyTorch's autograd system.

Let's illustrate this by rewriting our previous example using a class-based approach:

```python
import torch
import torch.nn as nn

class CustomNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Create an instance of the CustomNetwork
model = CustomNetwork()
```

In this example, we define a class called `CustomNetwork` that inherits from `nn.Module`. Inside the class, we describe the network's layers as attributes. The `forward` method specifies how the input flows through these layers during computation.

By defining neural networks as classes, we can create more complex architectures, leverage conditional logic, and encapsulate reusable components. This flexibility becomes particularly useful as we delve into more advanced topics.

## Choosing the Appropriate Method

While both the `nn.Sequential` and class-based approaches allow us to define neural networks, we will primarily utilize `nn.Sequential` in this course due to its simplicity and convenience. It offers an efficient way to construct networks with a linear sequence of layers.

However, understanding the class-based approach is crucial for deeper customization and the construction of more intricate architectures. As you progress with PyTorch, you will encounter situations where defining networks as classes is necessary.

In the upcoming chapters, we will explore various applications of `nn.Sequential` and learn advanced techniques to enhance the performance and capabilities of our neural networks.

The following code presents a complete example of the MPG dataset using an `nn.Module` class.


In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler


class Net(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        # Define each of the layers
        self.layer1 = nn.Linear(input_dim, 50)
        self.layer2 = nn.Linear(50, 25)
        self.layer3 = nn.Linear(25, output_dim)

    def forward(self, x):
        # Pass the input through each of the layers
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)


# Load and preprocess data
df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv",
    na_values=["NA", "?"],
)

# Replace missing horsepower values with median
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

# One-hot encode categorical column
df = pd.get_dummies(df, columns=["origin"], dtype=float)

# Split features/target
x_np = df[
    [
        "cylinders",
        "displacement",
        "horsepower",
        "weight",
        "acceleration",
        "year",
        "origin_1",
        "origin_2",
        "origin_3",
    ]
].values

y_np = df["mpg"].values

# Scale features
scaler = StandardScaler()
x_np = scaler.fit_transform(x_np)

# Convert to PyTorch tensors
x = torch.tensor(x_np, device=device, dtype=torch.float32)
y = torch.tensor(y_np, device=device, dtype=torch.float32)

# Initialize the model, loss function, and optimizer
model = Net(x.shape[1], 1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(1000):
    optimizer.zero_grad()

    outputs = model(x).flatten()
    loss = loss_fn(outputs, y)

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, loss: {loss.item()}")

Epoch 0, loss: 615.5946655273438
Epoch 100, loss: 7.534498691558838
Epoch 200, loss: 6.217309474945068
Epoch 300, loss: 5.76693868637085
Epoch 400, loss: 5.509234428405762
Epoch 500, loss: 5.293069839477539
Epoch 600, loss: 5.157623767852783
Epoch 700, loss: 5.042261600494385
Epoch 800, loss: 4.955922603607178
Epoch 900, loss: 4.896135330200195
